In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as ply
%matplotlib inline

In [5]:
words = open('names.txt', 'r').read().splitlines()
len(words)

32033

In [7]:
# Build mapping system
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [24]:
# Build the dataset
block_size = 3
X, Y = [], [] # Context and label
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
        print("".join(itos[i] for i in context), '--->', itos[ix])
X = torch.tensor(X)
Y = torch.tensor(Y)
print(f"\nX shape: {X.shape}")
print(f"Y shape: {Y.shape}")

emma
..e ---> e
.em ---> m
emm ---> m
mma ---> a
ma. ---> .
olivia
..o ---> o
.ol ---> l
oli ---> i
liv ---> v
ivi ---> i
via ---> a
ia. ---> .
ava
..a ---> a
.av ---> v
ava ---> a
va. ---> .
isabella
..i ---> i
.is ---> s
isa ---> a
sab ---> b
abe ---> e
bel ---> l
ell ---> l
lla ---> a
la. ---> .
sophia
..s ---> s
.so ---> o
sop ---> p
oph ---> h
phi ---> i
hia ---> a
ia. ---> .

X shape: torch.Size([32, 3])
Y shape: torch.Size([32])


Check Bengio et. Al, 2003 paper and Andej Karpathy's walkthrough to better clarify architecture. Key points:
- The model looks at 4 words, and aims to predict the 5th.
- Each word is given a 30 dimension vector embedding.
- Every (17000) word's embedding is stored inside a 17000 by 30 lookup matrix, C, which is used to map each of the 4 previous words to their embeddings, before being fed into the neural network to get predictions for the 5th word. 

For our experiment, we are going to make letter embeddings 2 dimensions, since there are only 27 possibilities compared to 17000 words in the paper.

In [37]:
C = torch.randn((27, 2)) # Lookup matrix contains a 2 dimension embedding for each of the 27 characters

In [38]:
C[X] # Lookup the embedding for each character in X
# The output is the embedding (2d) for each character in X (32 inputs x 3 characters x 2 dimensions)

tensor([[[-0.3708,  0.9240],
         [-0.3708,  0.9240],
         [-0.3708,  0.9240]],

        [[-0.3708,  0.9240],
         [-0.3708,  0.9240],
         [-0.0533,  0.1369]],

        [[-0.3708,  0.9240],
         [-0.0533,  0.1369],
         [ 0.0419, -0.4014]],

        [[-0.0533,  0.1369],
         [ 0.0419, -0.4014],
         [ 0.0419, -0.4014]],

        [[ 0.0419, -0.4014],
         [ 0.0419, -0.4014],
         [-2.4049, -0.3418]],

        [[-0.3708,  0.9240],
         [-0.3708,  0.9240],
         [-0.3708,  0.9240]],

        [[-0.3708,  0.9240],
         [-0.3708,  0.9240],
         [ 0.9631,  0.6704]],

        [[-0.3708,  0.9240],
         [ 0.9631,  0.6704],
         [-0.3781, -0.5619]],

        [[ 0.9631,  0.6704],
         [-0.3781, -0.5619],
         [ 0.2949,  0.5017]],

        [[-0.3781, -0.5619],
         [ 0.2949,  0.5017],
         [ 0.4749,  1.2226]],

        [[ 0.2949,  0.5017],
         [ 0.4749,  1.2226],
         [ 0.2949,  0.5017]],

        [[ 0.4749,  1

In [43]:
C[X].shape

torch.Size([32, 3, 2])

In [39]:
emb = C[X]

Hidden layer

In [40]:
W1 = torch.tensor((6,  100)) # 3x2 weights per neuron, 100 neurons

In [ ]:
# What we'd like to do:
# emb @ W1 + b1
# This won't work since emb is 32x3x2 and W1 is 6x100
# We must reshape emb from 32x3x2 into 32x6

In [41]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)

tensor([[-0.3708,  0.9240, -0.3708,  0.9240, -0.3708,  0.9240],
        [-0.3708,  0.9240, -0.3708,  0.9240, -0.0533,  0.1369],
        [-0.3708,  0.9240, -0.0533,  0.1369,  0.0419, -0.4014],
        [-0.0533,  0.1369,  0.0419, -0.4014,  0.0419, -0.4014],
        [ 0.0419, -0.4014,  0.0419, -0.4014, -2.4049, -0.3418],
        [-0.3708,  0.9240, -0.3708,  0.9240, -0.3708,  0.9240],
        [-0.3708,  0.9240, -0.3708,  0.9240,  0.9631,  0.6704],
        [-0.3708,  0.9240,  0.9631,  0.6704, -0.3781, -0.5619],
        [ 0.9631,  0.6704, -0.3781, -0.5619,  0.2949,  0.5017],
        [-0.3781, -0.5619,  0.2949,  0.5017,  0.4749,  1.2226],
        [ 0.2949,  0.5017,  0.4749,  1.2226,  0.2949,  0.5017],
        [ 0.4749,  1.2226,  0.2949,  0.5017, -2.4049, -0.3418],
        [-0.3708,  0.9240, -0.3708,  0.9240, -0.3708,  0.9240],
        [-0.3708,  0.9240, -0.3708,  0.9240, -2.4049, -0.3418],
        [-0.3708,  0.9240, -2.4049, -0.3418,  0.4749,  1.2226],
        [-2.4049, -0.3418,  0.4749,  1.2

In [42]:
torch.cat(torch.unbind(emb, 1), 1)

tensor([[-0.3708,  0.9240, -0.3708,  0.9240, -0.3708,  0.9240],
        [-0.3708,  0.9240, -0.3708,  0.9240, -0.0533,  0.1369],
        [-0.3708,  0.9240, -0.0533,  0.1369,  0.0419, -0.4014],
        [-0.0533,  0.1369,  0.0419, -0.4014,  0.0419, -0.4014],
        [ 0.0419, -0.4014,  0.0419, -0.4014, -2.4049, -0.3418],
        [-0.3708,  0.9240, -0.3708,  0.9240, -0.3708,  0.9240],
        [-0.3708,  0.9240, -0.3708,  0.9240,  0.9631,  0.6704],
        [-0.3708,  0.9240,  0.9631,  0.6704, -0.3781, -0.5619],
        [ 0.9631,  0.6704, -0.3781, -0.5619,  0.2949,  0.5017],
        [-0.3781, -0.5619,  0.2949,  0.5017,  0.4749,  1.2226],
        [ 0.2949,  0.5017,  0.4749,  1.2226,  0.2949,  0.5017],
        [ 0.4749,  1.2226,  0.2949,  0.5017, -2.4049, -0.3418],
        [-0.3708,  0.9240, -0.3708,  0.9240, -0.3708,  0.9240],
        [-0.3708,  0.9240, -0.3708,  0.9240, -2.4049, -0.3418],
        [-0.3708,  0.9240, -2.4049, -0.3418,  0.4749,  1.2226],
        [-2.4049, -0.3418,  0.4749,  1.2